# Stuttering Detection: Linear Classification Analysis
**Course**: CS204T (Artificial Intelligence)  
**Team**: 18  
**Focus**: Logistic Regression vs. The Perceptron

---

## Step 1: Environment Setup

In [2]:
import os
import shutil
import numpy as np
from src.extractors import WavLMExtractor
from src.data import DataManager
from src.models import LogisticModel, PerceptronModel

# Experiment Paths
AUDIO_DIR = "Stuttering Events in Podcasts Dataset/clips/stuttering-clips/clips"
CSV_PATHS = [
    "Stuttering Events in Podcasts Dataset/SEP-28k_labels.csv",
    "Stuttering Events in Podcasts Dataset/fluencybank_labels.csv"
]
FEATURE_DIR = "data/features"
fluent_dir = os.path.join(FEATURE_DIR, "fluent")
disfluent_dir = os.path.join(FEATURE_DIR, "disfluent")

/home/anshuman139/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 2 (Optional): Operational Mode for Data Extraction
* `SKIP_EXTRACTION`: Uses features already on disk (Default).
* `FORCE_EXTRACT`: Analyzes raw audio for new files (Resumable).
* `CLEAN_START`: Wipes the database and re-extracts from zero.

In [ ]:
# Operational Flags
SKIP_EXTRACTION = False
FORCE_EXTRACT = True
CLEAN_START = True
NUM_CLIPS_TO_EXTRACT = None

if CLEAN_START:
    if os.path.exists(FEATURE_DIR):
        shutil.rmtree(FEATURE_DIR)
    print("[System] Clean start initiated. Wiped feature database.")

if not SKIP_EXTRACTION or CLEAN_START or FORCE_EXTRACT:
    extractor = WavLMExtractor("microsoft/wavlm-base")
    label_dict = DataManager.generate_label_dict(CSV_PATHS, filter_quality=True)
    
    # Now using NATIVE Random Sampling logic for diversity
    extractor.extract_from_dir(
        AUDIO_DIR, 
        output_dir=FEATURE_DIR, 
        label_dict=label_dict, 
        limit=NUM_CLIPS_TO_EXTRACT, 
        random_sample=True
    )
else:
    print("[System] Skipping extraction. Using existing data on disk.")

[System] Clean start initiated. Wiped feature database.
[WavLMExtractor] Initializing on cpu...


/home/anshuman139/venv/lib/python3.12/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0
Loading weights: 100%|██████████████████████| 248/248 [00:00<00:00, 6893.91it/s]


[DataManager] Quality Filter: Removed 3938 low-quality samples.
[WavLMExtractor] Found 0 pre-existing features. Skipping redundant work.


Batch Extraction:   0%|                               | 0/32321 [00:00<?, ?it/s]/home/anshuman139/venv/lib/python3.12/site-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
Batch Extraction:   0%|                    | 15/32321 [00:28<2:48:11,  3.20it/s]

[WavLMExtractor] Error processing Stuttering Events in Podcasts Dataset/clips/stuttering-clips/clips/HeStutters_0_34.wav: Calculated padded input size per channel: (0). Kernel size: (10). Kernel size can't be greater than actual input size


Batch Extraction:   0%|                    | 85/32321 [00:47<2:16:42,  3.93it/s]

[WavLMExtractor] Error processing Stuttering Events in Podcasts Dataset/clips/stuttering-clips/clips/WomenWhoStutter_0_108.wav: Calculated padded input size per channel: (0). Kernel size: (10). Kernel size can't be greater than actual input size


Batch Extraction:   1%|                   | 205/32321 [01:20<2:26:35,  3.65it/s]

[WavLMExtractor] Error processing Stuttering Events in Podcasts Dataset/clips/stuttering-clips/clips/WomenWhoStutter_0_77.wav: Calculated padded input size per channel: (0). Kernel size: (10). Kernel size can't be greater than actual input size


Batch Extraction:   1%|▏                  | 247/32321 [01:32<2:34:05,  3.47it/s]

## Step 3: Data Loading and Preparation

In [3]:
label_dict = DataManager.generate_label_dict(CSV_PATHS, filter_quality=True)
manager = DataManager(None, None)

# Loading individual .npy files into training matrices
X, y = manager.load_from_folders(fluent_dir, disfluent_dir)
X_train, X_val, X_test, y_train, y_val, y_test = manager.get_splits(test_size=0.15, val_size=0.15)

# Oversampling to handle imbalance
X_train_bal, y_train_bal = manager.balance_data(X_train, y_train, strategy="oversample")

# Standardization
X_train_final = manager.preprocess(X_train_bal, method="standard")
X_test_final = manager.preprocess(X_test, method="standard")

[DataManager] Quality Filter: Removed 3938 low-quality samples.


## Step 4: Model 1 - Logistic Regression

In [4]:
log_model = LogisticModel("Logistic_Regression")
log_model.train(X_train_final, y_train_bal)
log_model.evaluate(X_test_final, y_test)

[Model: Logistic_Regression] Initialized.
[Logistic_Regression] Training on 28702 samples...

--- Evaluation: Logistic_Regression ---
Accuracy: 0.6967
Precision: 0.5562
Recall: 0.7493
F1: 0.6384

Confusion Matrix (Binary):
               Predicted: Fluent(0)  Predicted: Stutter(1)
True: Fluent(0)      2053            1023           
True: Stutter(1)     429             1282           


{'accuracy': 0.6966785042824316,
 'precision': 0.5561822125813449,
 'recall': 0.7492694330800701,
 'f1': 0.6384462151394422,
 'confusion_matrix': array([[2053, 1023],
        [ 429, 1282]])}

## Step 5: Model 2 - The Perceptron

In [6]:
perc_model = PerceptronModel("Iterative_Perceptron")
perc_model.train(X_train_final, y_train_bal)
perc_model.evaluate(X_test_final, y_test)

[Model: Iterative_Perceptron] Initialized.
[Iterative_Perceptron] Training on 28702 samples...

--- Evaluation: Iterative_Perceptron ---
Accuracy: 0.6282
Precision: 0.4849
Recall: 0.6464
F1: 0.5541

Confusion Matrix (Binary):
               Predicted: Fluent(0)  Predicted: Stutter(1)
True: Fluent(0)      1901            1175           
True: Stutter(1)     605             1106           


{'accuracy': 0.6281595989137246,
 'precision': 0.48487505480052606,
 'recall': 0.6464056107539451,
 'f1': 0.5541082164328658,
 'confusion_matrix': array([[1901, 1175],
        [ 605, 1106]])}